# Unit 5 Lecture 5: MDOF Modal Analysis
## Modal Decomposition & Forced Response

### Learning Objectives
1. Understand modal decomposition for MDOF systems
2. Apply mode superposition method
3. Analyze forced response using modal coordinates
4. Calculate Frequency Response Functions (FRF)
5. Understand modal damping and coupling

### Context
**Building on previous lectures**:
- L3: Multi-DOF systems, normal modes, eigenvalue problems
- L4: SDOF vibrations, damping, free response
- **This lecture**: Combine them - treat each mode as SDOF!

**Why modal analysis matters**:
- **Simplification**: n-DOF system -> n independent SDOF systems
- **Physical insight**: Understand how structure vibrates
- **Design**: Target specific modes for control
- **Testing**: Experimental modal analysis (EMA)

### Key Concepts

**MDOF equation of motion**:
$$\mathbf{M}\ddot{\mathbf{x}} + \mathbf{C}\dot{\mathbf{x}} + \mathbf{K}\mathbf{x} = \mathbf{F}(t)$$

**Modal transformation**:
$$\mathbf{x}(t) = \mathbf{\Phi}\mathbf{q}(t) = \sum_{i=1}^{n} \phi_i q_i(t)$$

where:
- $\mathbf{x}$ = physical coordinates (displacements)
- $\mathbf{q}$ = modal coordinates (mode amplitudes)
- $\mathbf{\Phi}$ = modal matrix (columns are mode shapes)
- $\phi_i$ = i-th mode shape
- $q_i(t)$ = i-th modal amplitude

**Decoupled modal equations** (with proportional damping):
$$\ddot{q}_i + 2\zeta_i\omega_i\dot{q}_i + \omega_i^2 q_i = \frac{\phi_i^T \mathbf{F}(t)}{m_i}$$

Each mode behaves as independent SDOF system!

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.linalg import eig
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Plot styling
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("Libraries imported successfully!")
print("Ready for modal analysis...")


---
## Part A: Modal Decomposition - The Key Idea

### Problem: MDOF systems are coupled

**2-DOF example**:
$$m_1\ddot{x}_1 + (k_1+k_2)x_1 - k_2x_2 = F_1(t)$$
$$m_2\ddot{x}_2 - k_2x_1 + (k_2+k_3)x_2 = F_2(t)$$

Motion of $x_1$ affects $x_2$ and vice versa (coupling terms: $-k_2x_2$, $-k_2x_1$)

### Solution: Transform to modal coordinates

**Step 1: Find natural frequencies and mode shapes**

Solve eigenvalue problem:
$$\det(\mathbf{K} - \omega^2\mathbf{M}) = 0 \quad \Rightarrow \quad \omega_1, \omega_2, ..., \omega_n$$

For each $\omega_i$, solve:
$$(\mathbf{K} - \omega_i^2\mathbf{M})\phi_i = 0 \quad \Rightarrow \quad \phi_i \text{ (mode shape)}$$

**Step 2: Normalize mode shapes**

Mass normalization:
$$\phi_i^T \mathbf{M} \phi_i = 1 \quad \text{(modal mass = 1)}$$

**Step 3: Orthogonality properties**

$$\phi_i^T \mathbf{M} \phi_j = \begin{cases} 1 & i=j \\ 0 & i\neq j \end{cases}$$
$$\phi_i^T \mathbf{K} \phi_j = \begin{cases} \omega_i^2 & i=j \\ 0 & i\neq j \end{cases}$$

These are the magic properties that decouple the system!

**Step 4: Transform coordinates**

$$\mathbf{x}(t) = \mathbf{\Phi}\mathbf{q}(t) = \phi_1 q_1(t) + \phi_2 q_2(t) + ... + \phi_n q_n(t)$$

Substitute into original equation:
$$\mathbf{M}\mathbf{\Phi}\ddot{\mathbf{q}} + \mathbf{C}\mathbf{\Phi}\dot{\mathbf{q}} + \mathbf{K}\mathbf{\Phi}\mathbf{q} = \mathbf{F}$$

Multiply by $\mathbf{\Phi}^T$:
$$\mathbf{\Phi}^T\mathbf{M}\mathbf{\Phi}\ddot{\mathbf{q}} + \mathbf{\Phi}^T\mathbf{C}\mathbf{\Phi}\dot{\mathbf{q}} + \mathbf{\Phi}^T\mathbf{K}\mathbf{\Phi}\mathbf{q} = \mathbf{\Phi}^T\mathbf{F}$$

With orthogonality:
$$\mathbf{I}\ddot{\mathbf{q}} + \text{diag}(2\zeta_i\omega_i)\dot{\mathbf{q}} + \text{diag}(\omega_i^2)\mathbf{q} = \mathbf{\Phi}^T\mathbf{F}$$

**Result: n independent SDOF equations!**
$$\ddot{q}_i + 2\zeta_i\omega_i\dot{q}_i + \omega_i^2 q_i = f_i(t)$$

where $f_i(t) = \phi_i^T \mathbf{F}(t)$ = modal force

### Physical Interpretation

**General motion** = sum of modes vibrating independently:
- Mode 1 vibrates at $\omega_1$ with amplitude $q_1(t)$
- Mode 2 vibrates at $\omega_2$ with amplitude $q_2(t)$
- Total motion: $\mathbf{x}(t) = \phi_1 q_1(t) + \phi_2 q_2(t)$

Each mode acts like independent spring-mass system!

---


In [ ]:
# Example 1: Modal Decomposition of 2-DOF System

print("=" * 60)
print("Example 1: Modal Decomposition - Free Vibration")
print("=" * 60)

# System properties (same as L3)
m1, m2 = 1.0, 1.5  # kg
k1, k2, k3 = 100.0, 150.0, 100.0  # N/m

# Matrices
M = np.array([[m1, 0], [0, m2]])
K = np.array([[k1+k2, -k2], [-k2, k2+k3]])

print(f"\nSystem Properties:")
print(f"m1 = {m1} kg, m2 = {m2} kg")
print(f"k1 = {k1} N/m, k2 = {k2} N/m, k3 = {k3} N/m")

# Solve eigenvalue problem
M_inv = np.linalg.inv(M)
A = M_inv @ K
eigenvalues, eigenvectors = np.linalg.eig(A)

# Sort by frequency
idx = np.argsort(eigenvalues)
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

omega_n = np.sqrt(eigenvalues)
f_n = omega_n / (2 * np.pi)

print(f"\nNatural Frequencies:")
for i, (omega, f) in enumerate(zip(omega_n, f_n), 1):
    print(f"Mode {i}: omega_{i} = {omega:.3f} rad/s, f_{i} = {f:.3f} Hz")

# Mass-normalize mode shapes
mode_shapes = np.zeros_like(eigenvectors)
for i in range(len(omega_n)):
    phi = eigenvectors[:, i]
    # Mass normalization: phi^T * M * phi = 1
    modal_mass = phi.T @ M @ phi
    mode_shapes[:, i] = phi / np.sqrt(modal_mass)

print(f"\nMass-Normalized Mode Shapes:")
for i in range(len(omega_n)):
    print(f"Mode {i+1}: [{mode_shapes[0,i]:.4f}, {mode_shapes[1,i]:.4f}]")

# Verify orthogonality
print(f"\nOrthogonality Check:")
for i in range(len(omega_n)):
    for j in range(len(omega_n)):
        phi_i = mode_shapes[:, i]
        phi_j = mode_shapes[:, j]
        m_ij = phi_i.T @ M @ phi_j
        k_ij = phi_i.T @ K @ phi_j
        if i == j:
            print(f"phi_{i+1}^T M phi_{j+1} = {m_ij:.6f} (should be 1.0)")
            print(f"phi_{i+1}^T K phi_{j+1} = {k_ij:.6f} (should be {omega_n[i]**2:.3f})")
        else:
            print(f"phi_{i+1}^T M phi_{j+1} = {m_ij:.6f} (should be 0.0)")

# Simulate in modal coordinates
def modal_ode(q, t, omega_n, zeta):
    """Decoupled modal equations"""
    n = len(omega_n)
    q_pos = q[:n]
    q_vel = q[n:]
    
    q_acc = np.zeros(n)
    for i in range(n):
        q_acc[i] = -2*zeta[i]*omega_n[i]*q_vel[i] - omega_n[i]**2*q_pos[i]
    
    return np.concatenate([q_vel, q_acc])

# Initial conditions in physical coordinates
x0_phys = np.array([0.1, 0.0])  # m1 displaced, m2 at rest
v0_phys = np.array([0.0, 0.0])  # released from rest

# Transform to modal coordinates
# x = Phi * q  =>  q = Phi^T * M * x  (for mass-normalized modes)
q0 = mode_shapes.T @ M @ x0_phys
q_dot0 = mode_shapes.T @ M @ v0_phys

print(f"\nInitial Conditions:")
print(f"Physical: x = [{x0_phys[0]:.3f}, {x0_phys[1]:.3f}] m")
print(f"Modal: q = [{q0[0]:.6f}, {q0[1]:.6f}]")
print(f"-> Mode 1 excited more than Mode 2!")

# Simulate (undamped)
zeta = np.array([0.0, 0.0])  # No damping
t = np.linspace(0, 10, 2000)
y0 = np.concatenate([q0, q_dot0])
solution = odeint(modal_ode, y0, t, args=(omega_n, zeta))

# Extract modal coordinates
q1 = solution[:, 0]
q2 = solution[:, 1]

# Transform back to physical coordinates
x1 = mode_shapes[0, 0] * q1 + mode_shapes[0, 1] * q2
x2 = mode_shapes[1, 0] * q1 + mode_shapes[1, 1] * q2

# Plotting
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Physical coordinates
ax1 = axes[0, 0]
ax1.plot(t, x1*100, 'b-', linewidth=1.5, label='x1 (mass 1)')
ax1.plot(t, x2*100, 'r-', linewidth=1.5, label='x2 (mass 2)')
ax1.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax1.set_xlabel('Time (s)', fontsize=11)
ax1.set_ylabel('Displacement (cm)', fontsize=11)
ax1.set_title('Physical Coordinates (Coupled)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim([0, 10])

# Modal coordinates
ax2 = axes[0, 1]
ax2.plot(t, q1, 'g-', linewidth=1.5, label=f'q1 (Mode 1: {f_n[0]:.2f} Hz)')
ax2.plot(t, q2, 'm-', linewidth=1.5, label=f'q2 (Mode 2: {f_n[1]:.2f} Hz)')
ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax2.set_xlabel('Time (s)', fontsize=11)
ax2.set_ylabel('Modal Amplitude', fontsize=11)
ax2.set_title('Modal Coordinates (Decoupled!)', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, 10])

# FFT of physical coordinates
ax3 = axes[1, 0]
from scipy.fft import fft, fftfreq
dt = t[1] - t[0]
freq = fftfreq(len(t), dt)[:len(t)//2]
X1_fft = np.abs(fft(x1))[:len(t)//2]
X1_fft = X1_fft / np.max(X1_fft)  # Normalize
ax3.plot(freq, X1_fft, 'b-', linewidth=1.5)
ax3.axvline(x=f_n[0], color='g', linestyle='--', linewidth=2, alpha=0.7, label=f'f1={f_n[0]:.2f} Hz')
ax3.axvline(x=f_n[1], color='m', linestyle='--', linewidth=2, alpha=0.7, label=f'f2={f_n[1]:.2f} Hz')
ax3.set_xlabel('Frequency (Hz)', fontsize=11)
ax3.set_ylabel('Amplitude (normalized)', fontsize=11)
ax3.set_title('FFT: Two Natural Frequencies!', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim([0, 5])

# Mode shapes visualization
ax4 = axes[1, 1]
x_pos = np.array([0, 1, 2])  # Position: wall, m1, m2
for i in range(len(omega_n)):
    mode_viz = np.array([0, mode_shapes[0, i], mode_shapes[1, i]])
    ax4.plot(x_pos, mode_viz, 'o-', linewidth=2, markersize=10, label=f'Mode {i+1}')
ax4.axhline(y=0, color='k', linestyle='-', alpha=0.3)
ax4.set_xlabel('Mass Position', fontsize=11)
ax4.set_ylabel('Modal Amplitude', fontsize=11)
ax4.set_title('Mode Shapes', fontsize=12, fontweight='bold')
ax4.set_xticks([0, 1, 2])
ax4.set_xticklabels(['Wall', 'm1', 'm2'])
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nKey observations:")
print(f"1. Physical coords: Complex beating pattern (two frequencies mixed)")
print(f"2. Modal coords: Pure sinusoids at natural frequencies!")
print(f"3. FFT shows two peaks at f1 and f2")
print(f"4. Mode shapes independent (orthogonal)")
print(f"5. General motion = sum of modal motions")


---
## Part B: Forced Response - Resonance in MDOF Systems

### Harmonic Forcing

**Applied force**: $\mathbf{F}(t) = \mathbf{F}_0 \sin(\Omega t)$

where $\Omega$ = forcing frequency (rad/s)

### Modal equation for mode i:

$$\ddot{q}_i + 2\zeta_i\omega_i\dot{q}_i + \omega_i^2 q_i = \frac{\phi_i^T \mathbf{F}_0}{m_i} \sin(\Omega t)$$

Define modal force amplitude: $F_{i,0} = \phi_i^T \mathbf{F}_0$

### Steady-state response (SDOF theory!):

$$q_i(t) = Q_i \sin(\Omega t - \psi_i)$$

where:
$$Q_i = \frac{F_{i,0}/\omega_i^2}{\sqrt{(1-r_i^2)^2 + (2\zeta_i r_i)^2}}$$

$$\psi_i = \arctan\left(\frac{2\zeta_i r_i}{1-r_i^2}\right)$$

$$r_i = \frac{\Omega}{\omega_i} \quad \text{(frequency ratio)}$$

### Physical response:

$$\mathbf{x}(t) = \sum_{i=1}^{n} \phi_i Q_i \sin(\Omega t - \psi_i)$$

### Resonance Behavior

**When $\Omega \approx \omega_i$** (forcing near natural frequency):
- Mode $i$ amplitude $Q_i$ becomes very large!
- Amplification factor: $Q_i \approx \frac{1}{2\zeta_i}$ at resonance
- Other modes contribute less

**Multiple resonances**:
- n-DOF system has n resonance peaks
- Each corresponds to one natural frequency

### Frequency Response Function (FRF)

**Definition**: Response amplitude vs. forcing frequency

For excitation at DOF j, response at DOF i:
$$H_{ij}(\Omega) = \frac{X_i(\Omega)}{F_j(\Omega)} = \sum_{k=1}^{n} \frac{\phi_{ik}\phi_{jk}}{\omega_k^2 - \Omega^2 + 2i\zeta_k\omega_k\Omega}$$

**FRF magnitude** shows all natural frequencies as peaks!

**Used in**:
- Modal testing (hammer test)
- Vibration isolation design
- Structural health monitoring

---


In [ ]:
# Example 2: Forced Response - Sweeping Through Resonances

print("=" * 60)
print("Example 2: Forced Harmonic Response")
print("=" * 60)

# Same system, add damping
zeta = np.array([0.05, 0.05])  # 5% damping in each mode

print(f"\nSystem with damping:")
print(f"zeta_1 = {zeta[0]}, zeta_2 = {zeta[1]}")
print(f"omega_1 = {omega_n[0]:.3f} rad/s, omega_2 = {omega_n[1]:.3f} rad/s")

# Forcing on mass 1
F0_phys = np.array([1.0, 0.0])  # 1 N on m1, nothing on m2

# Modal forces
F0_modal = np.zeros(len(omega_n))
for i in range(len(omega_n)):
    F0_modal[i] = mode_shapes[:, i].T @ F0_phys

print(f"\nForcing:")
print(f"Physical: F = [1.0, 0.0] N (on m1 only)")
print(f"Modal: F_modal = [{F0_modal[0]:.4f}, {F0_modal[1]:.4f}]")

# Frequency sweep
Omega_range = np.linspace(0.1, 25, 500)  # rad/s

# Calculate steady-state response amplitude for each frequency
X1_amp = np.zeros(len(Omega_range))
X2_amp = np.zeros(len(Omega_range))

for idx, Omega in enumerate(Omega_range):
    # Modal response amplitudes
    Q = np.zeros(len(omega_n))
    for i in range(len(omega_n)):
        r = Omega / omega_n[i]
        denominator = np.sqrt((1 - r**2)**2 + (2*zeta[i]*r)**2)
        Q[i] = (F0_modal[i] / omega_n[i]**2) / denominator
    
    # Physical response (sum of modal contributions)
    x_response = mode_shapes @ Q
    X1_amp[idx] = abs(x_response[0])
    X2_amp[idx] = abs(x_response[1])

# Convert to frequency in Hz
freq_range = Omega_range / (2 * np.pi)

# Plotting
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# FRF for mass 1
ax1 = axes[0]
ax1.semilogy(freq_range, X1_amp*100, 'b-', linewidth=2, label='X1 (mass 1)')
ax1.axvline(x=f_n[0], color='g', linestyle='--', linewidth=2, alpha=0.7, label=f'f1={f_n[0]:.2f} Hz')
ax1.axvline(x=f_n[1], color='m', linestyle='--', linewidth=2, alpha=0.7, label=f'f2={f_n[1]:.2f} Hz')
ax1.set_xlabel('Forcing Frequency (Hz)', fontsize=11)
ax1.set_ylabel('Response Amplitude (cm/N)', fontsize=11)
ax1.set_title('FRF: H11 - Force on m1, Response of m1 (Two Resonance Peaks!)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, which='both')
ax1.set_xlim([0, 4])

# FRF for mass 2
ax2 = axes[1]
ax2.semilogy(freq_range, X2_amp*100, 'r-', linewidth=2, label='X2 (mass 2)')
ax2.axvline(x=f_n[0], color='g', linestyle='--', linewidth=2, alpha=0.7, label=f'f1={f_n[0]:.2f} Hz')
ax2.axvline(x=f_n[1], color='m', linestyle='--', linewidth=2, alpha=0.7, label=f'f2={f_n[1]:.2f} Hz')
ax2.set_xlabel('Forcing Frequency (Hz)', fontsize=11)
ax2.set_ylabel('Response Amplitude (cm/N)', fontsize=11)
ax2.set_title('FRF: H21 - Force on m1, Response of m2', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, which='both')
ax2.set_xlim([0, 4])

plt.tight_layout()
plt.show()

# Find peak amplifications
idx_peak1 = np.argmax(X1_amp[freq_range < f_n[0]*1.2])
idx_peak2 = np.argmax(X1_amp[freq_range > f_n[0]*1.2])
peak1_amp = X1_amp[idx_peak1]
peak2_amp = X1_amp[idx_peak2]

print(f"\nResonance Analysis:")
print(f"Peak 1 (at f1={f_n[0]:.2f} Hz): Amplitude = {peak1_amp*100:.2f} cm/N")
print(f"Peak 2 (at f2={f_n[1]:.2f} Hz): Amplitude = {peak2_amp*100:.2f} cm/N")
print(f"Amplification factor 1: {peak1_amp / (F0_phys[0]/k1):.1f}x")
print(f"Theoretical (Q=1/(2*zeta)): {1/(2*zeta[0]):.1f}x")

print(f"\nKey observations:")
print(f"1. Two resonance peaks at natural frequencies")
print(f"2. Peak heights controlled by damping")
print(f"3. Between peaks: reduced response (anti-resonance)")
print(f"4. At low frequency: quasi-static response")
print(f"5. FRF is fingerprint of structure!")


In [ ]:
# Example 3: Time-domain simulation at resonance

print("=" * 60)
print("Example 3: Time Response at Resonance")
print("=" * 60)

# Excite at first natural frequency
Omega_excite = omega_n[0]  # First natural frequency
F_amp = 1.0  # N

print(f"\nExcitation:")
print(f"Frequency: Omega = {Omega_excite:.3f} rad/s = {Omega_excite/(2*np.pi):.3f} Hz")
print(f"Matches natural frequency 1! (resonance)")
print(f"Amplitude: F = {F_amp} N on mass 1")

def modal_forced_ode(q, t, omega_n, zeta, F_modal, Omega):
    """Modal equations with forcing"""
    n = len(omega_n)
    q_pos = q[:n]
    q_vel = q[n:]
    
    q_acc = np.zeros(n)
    for i in range(n):
        # Forcing term
        f_modal = F_modal[i] * np.sin(Omega * t)
        q_acc[i] = -2*zeta[i]*omega_n[i]*q_vel[i] - omega_n[i]**2*q_pos[i] + f_modal
    
    return np.concatenate([q_vel, q_acc])

# Initial conditions (at rest)
q0 = np.zeros(len(omega_n))
q_dot0 = np.zeros(len(omega_n))
y0 = np.concatenate([q0, q_dot0])

# Simulate
t_sim = np.linspace(0, 20, 4000)
F_modal_amp = np.zeros(len(omega_n))
for i in range(len(omega_n)):
    F_modal_amp[i] = mode_shapes[:, i].T @ (F_amp * F0_phys)

solution = odeint(modal_forced_ode, y0, t_sim, args=(omega_n, zeta, F_modal_amp, Omega_excite))

# Extract modal coordinates
q1_forced = solution[:, 0]
q2_forced = solution[:, 1]

# Transform to physical
x1_forced = mode_shapes[0, 0] * q1_forced + mode_shapes[0, 1] * q2_forced
x2_forced = mode_shapes[1, 0] * q1_forced + mode_shapes[1, 1] * q2_forced

# Plotting
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Physical response
ax1 = axes[0]
ax1.plot(t_sim, x1_forced*100, 'b-', linewidth=1.5, label='x1 (mass 1)')
ax1.plot(t_sim, x2_forced*100, 'r-', linewidth=1.5, label='x2 (mass 2)')
# Show forcing
forcing_viz = F_amp * np.sin(Omega_excite * t_sim) * 5  # Scaled for visibility
ax1.plot(t_sim, forcing_viz, 'k--', linewidth=1, alpha=0.5, label='Force (scaled)')
ax1.set_xlabel('Time (s)', fontsize=11)
ax1.set_ylabel('Displacement (cm)', fontsize=11)
ax1.set_title('Physical Response: Build-up at Resonance', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Modal coordinates
ax2 = axes[1]
ax2.plot(t_sim, q1_forced, 'g-', linewidth=1.5, label='q1 (Mode 1 - RESONANT!)')
ax2.plot(t_sim, q2_forced, 'm-', linewidth=1.5, label='q2 (Mode 2 - minimal)')
ax2.set_xlabel('Time (s)', fontsize=11)
ax2.set_ylabel('Modal Amplitude', fontsize=11)
ax2.set_title('Modal Response: Mode 1 Dominates!', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Amplitude envelope (mass 1)
ax3 = axes[2]
from scipy.signal import hilbert
envelope = np.abs(hilbert(x1_forced))
ax3.plot(t_sim, x1_forced*100, 'b-', linewidth=1, alpha=0.5, label='x1')
ax3.plot(t_sim, envelope*100, 'r-', linewidth=2, label='Envelope (build-up)')
ax3.plot(t_sim, -envelope*100, 'r-', linewidth=2)
ax3.set_xlabel('Time (s)', fontsize=11)
ax3.set_ylabel('Displacement (cm)', fontsize=11)
ax3.set_title('Resonance Build-up: Exponential Growth to Steady-State', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate steady-state amplitude
steady_amp = np.max(envelope[t_sim > 15])  # After transient
theoretical_amp = (F_modal_amp[0] / omega_n[0]**2) / (2 * zeta[0])

print(f"\nSteady-State Analysis:")
print(f"Measured amplitude: {steady_amp*100:.2f} cm")
print(f"Theoretical (resonance): {theoretical_amp*100:.2f} cm")
print(f"Build-up time constant: tau ~ 1/(zeta*omega_n) = {1/(zeta[0]*omega_n[0]):.2f} s")

print(f"\nKey observations:")
print(f"1. Exponential build-up to steady-state")
print(f"2. Mode 1 amplitude grows large (resonance!)")
print(f"3. Mode 2 stays small (off-resonance)")
print(f"4. Build-up rate depends on damping")
print(f"5. Lower damping -> higher peak, slower build-up")


---
## Summary

### Key Concepts

**1. Modal Decomposition**
- Transform from physical coordinates x to modal coordinates q
- Decouples equations: n-DOF -> n independent SDOF
- Each mode vibrates independently at its natural frequency

**2. Mode Superposition**
- General motion = sum of modal contributions
- $\\mathbf{x}(t) = \\sum \\phi_i q_i(t)$
- Modal amplitudes evolve according to SDOF equations

**3. Forced Response**
- Apply SDOF forced response to each mode
- Resonance occurs when forcing frequency matches natural frequency
- n natural frequencies -> n resonance peaks in FRF

**4. Frequency Response Function**
- Shows response amplitude vs. frequency
- Peaks at natural frequencies (resonances)
- Used in modal testing and system identification

### Modal Analysis Procedure

**For free vibration**:
1. Form M and K matrices
2. Solve eigenvalue problem: det(K - omega^2 M) = 0
3. Get natural frequencies omega_i and mode shapes phi_i
4. Transform initial conditions to modal coordinates
5. Solve decoupled SDOF equations
6. Transform back to physical coordinates

**For forced vibration**:
1. Calculate modal forces: F_modal = Phi^T * F
2. Solve each modal equation (SDOF forced response)
3. Sum modal responses to get physical response

### Design Implications

**Avoid resonance**:
- Keep forcing frequencies away from natural frequencies
- If unavoidable, add damping
- Change mass or stiffness to shift natural frequencies

**Modal control**:
- Target specific modes for vibration suppression
- Add damping preferentially to troublesome modes
- Tune mass dampers to specific frequencies

**Testing strategy**:
- Hammer test excites all modes
- Measure FRF to identify natural frequencies
- Extract mode shapes from multiple measurements
- Compare with finite element model

### Connection to Real Systems

**Examples**:
- **Buildings**: Ground motion (earthquake) excites multiple modes
- **Bridges**: Wind or traffic can excite resonances
- **Machinery**: Rotating imbalance creates harmonic forcing
- **Aircraft**: Turbulence excites structural modes

### Limitations

**Modal analysis assumes**:
- Linear system (superposition valid)
- Proportional damping (modes decouple)
- Time-invariant properties

**When it fails**:
- Large deformations (geometric nonlinearity)
- Material nonlinearity
- Non-proportional damping (modes couple through damping)
- Time-varying systems

### What's Next?

**Unit 5 Practical**: Modal Testing Simulation
- Hammer test simulation
- FRF measurement
- Parameter extraction
- Modal assurance criterion (MAC)

**Then**: Unit 6 (Work & Energy) and Unit 7 (Numerical Methods)

---

### Practice Problems

1. **3-DOF system**: Three equal masses in series, find all three modes
2. **Modal contribution**: Force on middle mass, which modes are excited?
3. **Damping design**: Add damper to reduce resonance peak by 50%
4. **FRF analysis**: Given measured FRF, extract omega_n and zeta

---
